# Assignment 1 - LLM Evaluation

End-to-end LLM evaluation pipeline: generate product descriptions, evaluate manually and with an LLM judge, iterate to improve.

## Setup

In [9]:
import sys
sys.path.insert(0, '.')

import pandas as pd
from pathlib import Path

from src.config import OPENAI_API_KEY, GENERATOR_MODEL, JUDGE_MODEL
from src.rubric import RUBRIC, ALL_CRITERIA, JUDGE_CRITERIA, compute_final_score, rate_length, count_words
from src.generator import generate_description, SYSTEM_PROMPT, build_user_prompt
from src.judge import judge_all_criteria, judge_single_criterion

print(f"Generator model: {GENERATOR_MODEL}")
print(f"Judge model: {JUDGE_MODEL}")
print(f"API key loaded: {'Yes' if OPENAI_API_KEY else 'No'}")

Generator model: gpt-4o-mini
Judge model: gpt-4o
API key loaded: Yes


In [10]:
# Load dataset
DATASET_PATH = Path('Assignment_01_product_dataset.xlsx')
df_products = pd.read_excel(DATASET_PATH)
print(f"Products loaded: {len(df_products)}")
df_products.head(3)

Products loaded: 50


,product_name,Product_attribute_list,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty


---
## Task 1 - Define Your Rubric (15 pts)

The rubric is defined in `src/rubric.py`. Let's display it.

In [12]:
# Display rubric definitions
rubric_rows = []
for criterion, ratings in RUBRIC.items():
    rubric_rows.append({
        'Criterion': criterion.title(),
        'Good': ratings['good'],
        'Ok': ratings['ok'],
        'Bad': ratings['bad'],
    })
df_rubric = pd.DataFrame(rubric_rows)
df_rubric.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})

,Criterion,Good,Ok,Bad
0,Fluency,"Natural, easy-to-read sentences that flow well. No awkward phrasing.",Readable but contains some awkward phrasing or unnatural word choices.,"Hard to read, broken or incomplete sentences, incoherent structure."
1,Grammar,No spelling or punctuation errors.,1-2 minor spelling or punctuation errors.,"3 or more errors, or any major grammatical mistake."
2,Tone,"Friendly, credible sales voice. Persuasive without being pushy.",Mostly appropriate tone but inconsistent — shifts between formal/casual.,"Wrong tone entirely: too formal, robotic, aggressive, or sarcastic."
3,Length,50-90 words.,40-49 or 91-110 words.,Below 40 or above 110 words.
4,Grounding,Uses only information provided in the product features. No invented claims.,Mostly grounded but includes minor embellishments or vague generalizations.,"Invents features, specs, or claims not present in the input data."
5,Latency,Average time per call < 2 seconds.,Average time per call 2-5 seconds.,Average time per call > 5 seconds.
6,Cost,Average cost per call < $0.001.,Average cost per call $0.001-$0.005.,Average cost per call > $0.005.


### Pass / Fail Definition

**Cumulative pass bar:** At least 4 'good' ratings and 0 'bad' ratings.

**Go / no-go rule:** If Grounding is not 'good', the description is automatically rejected (fail), regardless of other scores.

---
## Task 2 - Generate Descriptions (20 pts)

In [13]:
# Display the system prompt
print("System Prompt:")
print(SYSTEM_PROMPT)
print()
print("Example user prompt:")
print(build_user_prompt(df_products.iloc[0].to_dict()))

System Prompt:
You are an expert e-commerce copywriter. Write a persuasive product description based ONLY on the provided product information. The description must be between 50 and 90 words. Use a friendly, credible sales voice. Do not invent features or claims not present in the input. Output ONLY the description text, nothing else.

Example user prompt:
Product: Apple iPhone 15 Pro
Attributes: features: A17 Pro chip, 120 Hz ProMotion display, USB‑C fast charging; dimensions: compact
Material: titanium frame, Ceramic Shield glass
Warranty: 1‑year limited warranty


In [19]:
# Generate descriptions for all products
results = []
for idx, row in df_products.iterrows():
    product = row.to_dict()
    print(f"Generating {idx+1}/{len(df_products)}: {product['product_name']}...", end=' ')
    result = generate_description(product)
    result['product_name'] = product['product_name']
    result['word_count'] = count_words(result['generated_description'])
    results.append(result)
    print(f"done ({result['word_count']} words, {result['latency_ms']:.0f}ms)")

df_results = pd.DataFrame(results)
print(f"\nGenerated {len(df_results)} descriptions")
print(f"Avg latency: {df_results['latency_ms'].mean():.0f}ms")
print(f"Avg word count: {df_results['word_count'].mean():.1f}")


Generating 1/50: Apple iPhone 15 Pro... done (78 words, 3290ms)
Generating 2/50: Samsung Galaxy S24 Ultra... done (83 words, 2650ms)
Generating 3/50: Google Pixel 8 Pro... done (74 words, 2311ms)
Generating 4/50: Sony WH‑1000XM5 Headphones... done (74 words, 2091ms)
Generating 5/50: Bose QuietComfort Ultra Earbuds... done (73 words, 2210ms)
Generating 6/50: Amazon Echo Dot (5th Gen)... done (76 words, 2133ms)
Generating 7/50: Dell XPS 13 9310 Laptop... done (75 words, 2691ms)
Generating 8/50: Apple MacBook Air 13″ (M3)... done (79 words, 2852ms)
Generating 9/50: Microsoft Surface Pro 10... done (75 words, 2152ms)
Generating 10/50: Garmin Forerunner 255... done (78 words, 2155ms)
Generating 11/50: Fitbit Charge 6... done (79 words, 2354ms)
Generating 12/50: GoPro HERO12 Black... done (72 words, 2213ms)
Generating 13/50: DJI Mini 4 Pro Drone... done (77 words, 2625ms)
Generating 14/50: Nintendo Switch OLED... done (76 words, 2461ms)
Generating 15/50: PlayStation 5 Slim... done (77 words,

In [20]:
df_results.head(3)


,generated_description,latency_ms,input_tokens,output_tokens,product_name,word_count
0,Experience the cutting-edge technology of the ...,3290.4,127,99,Apple iPhone 15 Pro,78
1,Experience the pinnacle of mobile technology w...,2650.1,125,101,Samsung Galaxy S24 Ultra,83
2,"Introducing the Google Pixel 8 Pro, where cutt...",2310.6,127,95,Google Pixel 8 Pro,74


In [21]:
# Merge with original product data and add blank evaluation columns
df = pd.concat([df_products, df_results[['generated_description', 'latency_ms', 'input_tokens', 'output_tokens', 'word_count']]], axis=1)

# Add blank columns for rubric criteria
for criterion in ALL_CRITERIA:
    df[criterion] = ''
df['final_score'] = ''

# Programmatically fill length rating
df['length'] = df['generated_description'].apply(rate_length)

# Save to Excel
OUTPUT_PATH = Path('assignment_01.xlsx')
df.to_excel(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
df[['product_name', 'generated_description', 'word_count', 'latency_ms', 'input_tokens', 'output_tokens']].head(5)

Saved to assignment_01.xlsx


,product_name,generated_description,word_count,latency_ms,input_tokens,output_tokens
0,Apple iPhone 15 Pro,Experience the cutting-edge technology of the ...,78,3290.4,127,99
1,Samsung Galaxy S24 Ultra,Experience the pinnacle of mobile technology w...,83,2650.1,125,101
2,Google Pixel 8 Pro,"Introducing the Google Pixel 8 Pro, where cutt...",74,2310.6,127,95
3,Sony WH‑1000XM5 Headphones,Experience unparalleled sound quality with the...,74,2091.3,125,100
4,Bose QuietComfort Ultra Earbuds,Experience unparalleled sound with the Bose Qu...,73,2209.7,117,96


---
## Task 3 - Manual (Human) Evaluation (10 pts)

In [22]:
# Task 3.1: Add cost column
# gpt-4o-mini pricing (as of 2025): $0.15/1M input, $0.60/1M output
INPUT_PRICE_PER_TOKEN = 0.15 / 1_000_000
OUTPUT_PRICE_PER_TOKEN = 0.60 / 1_000_000

df['cost_usd'] = (df['input_tokens'] * INPUT_PRICE_PER_TOKEN + 
                  df['output_tokens'] * OUTPUT_PRICE_PER_TOKEN)

print(f"Average cost per call: ${df['cost_usd'].mean():.6f}")
print(f"Total cost: ${df['cost_usd'].sum():.4f}")

# Rate cost and latency programmatically
avg_latency = df['latency_ms'].mean() / 1000  # convert to seconds
avg_cost = df['cost_usd'].mean()

latency_rating = 'good' if avg_latency < 2 else ('ok' if avg_latency <= 5 else 'bad')
cost_rating = 'good' if avg_cost < 0.001 else ('ok' if avg_cost <= 0.005 else 'bad')

df['latency'] = latency_rating
df['cost'] = cost_rating

print(f"\nAvg latency: {avg_latency:.2f}s -> {latency_rating}")
print(f"Avg cost: ${avg_cost:.6f} -> {cost_rating}")

Average cost per call: $0.000077
Total cost: $0.0038

Avg latency: 2.45s -> ok
Avg cost: $0.000077 -> good


In [23]:
# Task 3.2: Display first 15 products for manual evaluation
# Review these and fill in ratings below
MANUAL_EVAL_COUNT = 15

for idx in range(MANUAL_EVAL_COUNT):
    row = df.iloc[idx]
    print(f"=== Product {idx+1}: {row['product_name']} ===")
    print(f"Attributes: {row['Product_attribute_list']}")
    print(f"Material: {row['material']}")
    print(f"Warranty: {row['warranty']}")
    print(f"\nGenerated ({row['word_count']} words):")
    print(row['generated_description'])
    print(f"\nLength rating: {row['length']}")
    print('-' * 80)

=== Product 1: Apple iPhone 15 Pro ===
Attributes: features: A17 Pro chip, 120 Hz ProMotion display, USB‑C fast charging; dimensions: compact
Material: titanium frame, Ceramic Shield glass
Warranty: 1‑year limited warranty

Generated (78 words):
Experience the cutting-edge technology of the Apple iPhone 15 Pro. Powered by the A17 Pro chip, this compact device offers lightning-fast performance and a stunning 120 Hz ProMotion display for smooth visuals. Its durable titanium frame and Ceramic Shield glass ensure longevity and style, while USB-C fast charging keeps you connected in no time. With a 1-year limited warranty, you can enjoy peace of mind with your purchase. Elevate your smartphone experience with the iPhone 15 Pro today!

Length rating: good
--------------------------------------------------------------------------------
=== Product 2: Samsung Galaxy S24 Ultra ===
Attributes: features: 200 MP camera, S‑Pen support, 120 Hz AMOLED; sustainably sourced
Material: Armor Aluminum fra

In [26]:
# Task 3.2 & 3.3: Manual ratings for first 15 products
# Fill in your ratings after reviewing the descriptions above.
# For each product: rate fluency, grammar, tone, grounding as good/ok/bad
# (length, latency, cost are already rated programmatically)

# Example format - UPDATE THESE after reviewing:
manual_ratings = {
    # idx: {fluency, grammar, tone, grounding}
    0:  {'fluency': 'ok', 'grammar': 'good', 'tone': 'good', 'grounding': 'good'},
    1:  {'fluency': 'good', 'grammar': 'good', 'tone': 'good', 'grounding': 'good'},
    2:  {'fluency': 'good', 'grammar': 'good', 'tone': 'ok', 'grounding': 'good'},
    3:  {'fluency': 'ok', 'grammar': 'good', 'tone': 'good', 'grounding': 'ok'},
    4:  {'fluency': 'good', 'grammar': 'bad', 'tone': 'good', 'grounding': 'good'},
    5:  {'fluency': 'good', 'grammar': 'good', 'tone': 'good', 'grounding': 'good'},
    6:  {'fluency': 'bad', 'grammar': 'good', 'tone': 'ok', 'grounding': 'ok'},
    7:  {'fluency': 'good', 'grammar': 'ok', 'tone': 'good', 'grounding': 'good'},
    8:  {'fluency': 'ok', 'grammar': 'good', 'tone': 'good', 'grounding': 'ok'},
    9:  {'fluency': 'good', 'grammar': 'good', 'tone': 'good', 'grounding': 'good'},
    10: {'fluency': 'good', 'grammar': 'good', 'tone': 'bad', 'grounding': 'good'},
    11: {'fluency': 'bad', 'grammar': 'ok', 'tone': 'good', 'grounding': 'ok'},
    12: {'fluency': 'good', 'grammar': 'good', 'tone': 'good', 'grounding': 'good'},
    13: {'fluency': 'good', 'grammar': 'good', 'tone': 'good', 'grounding': 'good'},
    14: {'fluency': 'good', 'grammar': 'good', 'tone': 'bad', 'grounding': 'bad'},
}

# Apply manual ratings
for idx, ratings in manual_ratings.items():
    for criterion, rating in ratings.items():
        df.at[idx, criterion] = rating

# Compute final_score for manually rated products
for idx in manual_ratings:
    row_ratings = {c: df.at[idx, c] for c in ALL_CRITERIA}
    df.at[idx, 'final_score'] = compute_final_score(row_ratings)

# Save
df.to_excel(OUTPUT_PATH, index=False)
print("Manual ratings applied and saved.")
df.loc[list(manual_ratings.keys()), ['product_name', 'fluency', 'grammar', 'tone', 'length', 'grounding', 'final_score']]

Manual ratings applied and saved.


,product_name,fluency,grammar,tone,length,grounding,final_score
0,Apple iPhone 15 Pro,ok,good,good,good,good,pass
1,Samsung Galaxy S24 Ultra,good,good,good,good,good,pass
2,Google Pixel 8 Pro,good,good,ok,good,good,pass
3,Sony WH‑1000XM5 Headphones,ok,good,good,good,ok,fail
4,Bose QuietComfort Ultra Earbuds,good,bad,good,good,good,fail
5,Amazon Echo Dot (5th Gen),good,good,good,good,good,pass
6,Dell XPS 13 9310 Laptop,bad,good,ok,good,ok,fail
7,Apple MacBook Air 13″ (M3),good,ok,good,good,good,pass
8,Microsoft Surface Pro 10,ok,good,good,good,ok,fail
9,Garmin Forerunner 255,good,good,good,good,good,pass


In [27]:
# Task 3.4: Baseline analysis
manual_indices = list(manual_ratings.keys())
criteria_for_analysis = ['fluency', 'grammar', 'tone', 'length', 'grounding']

print("=== Baseline Analysis ===")
print(f"Products evaluated: {len(manual_indices)}")
print(f"Pass rate: {(df.loc[manual_indices, 'final_score'] == 'pass').sum()}/{len(manual_indices)}")
print()

for criterion in criteria_for_analysis:
    values = df.loc[manual_indices, criterion]
    good_pct = (values == 'good').sum() / len(values) * 100
    ok_pct = (values == 'ok').sum() / len(values) * 100
    bad_pct = (values == 'bad').sum() / len(values) * 100
    print(f"{criterion:12s}: good={good_pct:5.1f}%  ok={ok_pct:5.1f}%  bad={bad_pct:5.1f}%")

print(f"\nLatency (all products): {latency_rating} (avg {avg_latency:.2f}s)")
print(f"Cost (all products): {cost_rating} (avg ${avg_cost:.6f})")

=== Baseline Analysis ===
Products evaluated: 15
Pass rate: 8/15

fluency     : good= 66.7%  ok= 20.0%  bad= 13.3%
grammar     : good= 80.0%  ok= 13.3%  bad=  6.7%
tone        : good= 73.3%  ok= 13.3%  bad= 13.3%
length      : good=100.0%  ok=  0.0%  bad=  0.0%
grounding   : good= 66.7%  ok= 26.7%  bad=  6.7%

Latency (all products): ok (avg 2.45s)
Cost (all products): good (avg $0.000077)


---
## Task 4 - Improvement Cycle (15 pts)

Based on baseline analysis, iterate to improve the weakest criteria.

In [28]:
# Experiment 1: Improved prompt with stricter constraints and a few-shot example
IMPROVED_SYSTEM_PROMPT = """\
You are an expert e-commerce copywriter. Write a persuasive product description.

RULES:
- Use ONLY the provided product information. Do NOT invent features or claims.
- Write EXACTLY 50-90 words.
- Use a friendly, credible sales voice — persuasive but not pushy.
- Use correct grammar and punctuation.
- Output ONLY the description text.

EXAMPLE:
Product: Sony WH-1000XM5
Attributes: features: industry-leading noise cancellation, 30hr battery, USB-C; weight: 250g
Material: soft-fit leather, lightweight plastic
Warranty: 1-year manufacturer warranty

Description: Experience unmatched serenity with the Sony WH-1000XM5 headphones. Featuring industry-leading noise cancellation and an impressive 30-hour battery life, these lightweight headphones keep you immersed in your music all day. The soft-fit leather cushions ensure lasting comfort at just 250g, while USB-C charging keeps things simple. Backed by a 1-year manufacturer warranty, these headphones deliver premium sound you can count on."""

print("Experiment 1: Improved prompt with few-shot example")
print("Changes: Added explicit rules, word count constraint, and one example")
print("Expected: Better grounding, more consistent length")

Experiment 1: Improved prompt with few-shot example
Changes: Added explicit rules, word count constraint, and one example
Expected: Better grounding, more consistent length


In [29]:
# Re-generate with improved prompt for the 15 manually evaluated products
from src.generator import client
import time

improved_results = []
for idx in manual_indices:
    product = df_products.iloc[idx].to_dict()
    user_prompt = build_user_prompt(product)
    
    start = time.time()
    response = client.chat.completions.create(
        model=GENERATOR_MODEL,
        messages=[
            {"role": "system", "content": IMPROVED_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.5,  # lower temperature for more consistent output
        max_tokens=200,
    )
    latency_ms = (time.time() - start) * 1000
    
    desc = response.choices[0].message.content.strip()
    wc = count_words(desc)
    improved_results.append({
        'idx': idx,
        'product_name': product['product_name'],
        'description': desc,
        'word_count': wc,
        'length_rating': rate_length(desc),
        'latency_ms': latency_ms,
    })
    print(f"{idx+1}. {product['product_name']}: {wc} words, length={rate_length(desc)}")

df_improved = pd.DataFrame(improved_results)
print(f"\nAvg word count: {df_improved['word_count'].mean():.1f}")
print(f"Length 'good' rate: {(df_improved['length_rating'] == 'good').sum()}/{len(df_improved)}")

1. Apple iPhone 15 Pro: 79 words, length=good
2. Samsung Galaxy S24 Ultra: 75 words, length=good
3. Google Pixel 8 Pro: 84 words, length=good
4. Sony WH‑1000XM5 Headphones: 68 words, length=good
5. Bose QuietComfort Ultra Earbuds: 74 words, length=good
6. Amazon Echo Dot (5th Gen): 73 words, length=good
7. Dell XPS 13 9310 Laptop: 82 words, length=good
8. Apple MacBook Air 13″ (M3): 73 words, length=good
9. Microsoft Surface Pro 10: 75 words, length=good
10. Garmin Forerunner 255: 72 words, length=good
11. Fitbit Charge 6: 78 words, length=good
12. GoPro HERO12 Black: 73 words, length=good
13. DJI Mini 4 Pro Drone: 75 words, length=good
14. Nintendo Switch OLED: 73 words, length=good
15. PlayStation 5 Slim: 78 words, length=good

Avg word count: 75.5
Length 'good' rate: 15/15


In [30]:
# Display improved descriptions for review
for _, row in df_improved.iterrows():
    print(f"=== {row['product_name']} ({row['word_count']} words) ===")
    print(row['description'])
    print()

=== Apple iPhone 15 Pro (79 words) ===
Discover the power of the Apple iPhone 15 Pro, featuring the groundbreaking A17 Pro chip for lightning-fast performance. Enjoy stunning visuals on a 120 Hz ProMotion display, making every swipe a delight. With USB-C fast charging, staying connected is easier than ever. Encased in a sleek titanium frame and protected by Ceramic Shield glass, this compact device combines durability with elegance. Plus, it comes with a 1-year limited warranty for your peace of mind. Elevate your mobile experience today!

=== Samsung Galaxy S24 Ultra (75 words) ===
Unleash your creativity with the Samsung Galaxy S24 Ultra. Equipped with a stunning 200 MP camera and S-Pen support, capturing every moment has never been easier. Enjoy vibrant visuals on the 120 Hz AMOLED display, all housed in a durable Armor Aluminum frame and Gorilla Glass Victus. Plus, with sustainably sourced materials, you can feel good about your choice. Backed by a 1-year limited warranty, this sma

In [31]:
# Document experiment results
print("=== Experiment 1 Summary ===")
print("What changed: Added explicit rules, word count constraint, few-shot example, lowered temperature to 0.5")
print("Why expected to help: Explicit constraints reduce hallucination, example shows desired format")
print(f"\nResults:")
print(f"  Length 'good' rate: {(df_improved['length_rating'] == 'good').sum()}/{len(df_improved)}")
print(f"  Avg word count: {df_improved['word_count'].mean():.1f} (target: 50-90)")
print("\n  >> Review descriptions above and compare quality to baseline.")
print("  >> Update manual_ratings if re-evaluating with improved descriptions.")

=== Experiment 1 Summary ===
What changed: Added explicit rules, word count constraint, few-shot example, lowered temperature to 0.5
Why expected to help: Explicit constraints reduce hallucination, example shows desired format

Results:
  Length 'good' rate: 15/15
  Avg word count: 75.5 (target: 50-90)

  >> Review descriptions above and compare quality to baseline.
  >> Update manual_ratings if re-evaluating with improved descriptions.


---
## Task 5 - Create a Judge Model (20 pts)

Using `gpt-4o` as judge (different from generator `gpt-4o-mini`).

The Pydantic schema places `explanation` before `verdict` — this forces the model to reason (chain-of-thought) before committing to a verdict, reducing snap judgments.

See `src/judge.py` for implementation.

In [32]:
# Display judge prompt
from src.judge import JUDGE_SYSTEM_PROMPT, JudgeOutput
print("Judge System Prompt:")
print(JUDGE_SYSTEM_PROMPT)
print("\nPydantic Schema:")
print(JudgeOutput.model_json_schema())

Judge System Prompt:
You are a strict product-description evaluator. You will be given:
1. The product information (name, attributes, material, warranty)
2. A generated description to evaluate

Rate the description on each criterion below using EXACTLY the definitions provided.
For each criterion, first explain your reasoning, then give a verdict (good / ok / bad).

## Rubric
### Fluency
  - good: Natural, easy-to-read sentences that flow well. No awkward phrasing.
  - ok: Readable but contains some awkward phrasing or unnatural word choices.
  - bad: Hard to read, broken or incomplete sentences, incoherent structure.
### Grammar
  - good: No spelling or punctuation errors.
  - ok: 1-2 minor spelling or punctuation errors.
  - bad: 3 or more errors, or any major grammatical mistake.
### Tone
  - good: Friendly, credible sales voice. Persuasive without being pushy.
  - ok: Mostly appropriate tone but inconsistent — shifts between formal/casual.
  - bad: Wrong tone entirely: too formal, 

### Why `explanation` before `verdict`?

Placing the explanation field before the verdict in the Pydantic schema forces the model to generate its reasoning **before** committing to a rating. This is a form of chain-of-thought prompting via output structure: the model must articulate why it's giving a certain rating before the rating itself appears in the output. This reduces the chance of snap judgments and makes the verdict more consistent with the reasoning.

---
## Task 6 - Run and Analyze the Judge (20 pts)

In [33]:
# Task 6.1: Sanity check on 5 products
print("=== Sanity Check: Judge on 5 products ===")
sanity_results = []
for idx in range(5):
    product = df_products.iloc[idx].to_dict()
    desc = df.at[idx, 'generated_description']
    
    print(f"\n--- Product {idx+1}: {product['product_name']} ---")
    print(f"Description: {desc[:100]}...")
    
    result = judge_all_criteria(product, desc)
    sanity_results.append({'idx': idx, **{c: result[c]['verdict'] for c in JUDGE_CRITERIA}})
    
    for criterion in JUDGE_CRITERIA:
        r = result[criterion]
        print(f"  {criterion:12s}: {r['verdict']:4s} | {r['explanation'][:80]}")

print("\n>> Review above. If verdicts seem wrong, adjust judge prompt in src/judge.py before proceeding.")

=== Sanity Check: Judge on 5 products ===

--- Product 1: Apple iPhone 15 Pro ---
Description: Experience the cutting-edge technology of the Apple iPhone 15 Pro. Powered by the A17 Pro chip, this...
  fluency     : good | The description is well-structured and easy to read. Sentences flow naturally wi
  grammar     : good | There are no spelling or punctuation errors in the description.
  tone        : good | The tone is friendly and persuasive, encouraging the reader to consider the prod
  length      : good | The description is 76 words long, which falls within the 50-90 word range.
  grounding   : good | The description accurately reflects the product information provided, without ad

--- Product 2: Samsung Galaxy S24 Ultra ---
Description: Experience the pinnacle of mobile technology with the Samsung Galaxy S24 Ultra. Capture stunning pho...
  fluency     : good | The description is well-structured and flows naturally. Each sentence transition
  grammar     : good | There are no sp

In [34]:
# Task 6.2: Full run - judge all 50 products
print("Running judge on all products...")
all_judge_results = []

for idx in range(len(df)):
    product = df_products.iloc[idx].to_dict()
    desc = df.at[idx, 'generated_description']
    
    result = judge_all_criteria(product, desc)
    row_result = {'idx': idx}
    for c in JUDGE_CRITERIA:
        row_result[f'judge_{c}'] = result[c]['verdict']
        row_result[f'judge_{c}_explanation'] = result[c]['explanation']
    all_judge_results.append(row_result)
    print(f"  {idx+1}/{len(df)}: {product['product_name']} -> "
          f"{', '.join(f'{c}={result[c]["verdict"]}' for c in JUDGE_CRITERIA)}")

df_judge = pd.DataFrame(all_judge_results)

# Add judge columns to main df
for c in JUDGE_CRITERIA:
    df[f'judge_{c}'] = df_judge[f'judge_{c}']
    df[f'judge_{c}_explanation'] = df_judge[f'judge_{c}_explanation']

# Compute judge final_score
for idx in range(len(df)):
    judge_ratings = {c: df.at[idx, f'judge_{c}'] for c in JUDGE_CRITERIA}
    judge_ratings['latency'] = df.at[idx, 'latency']
    judge_ratings['cost'] = df.at[idx, 'cost']
    df.at[idx, 'judge_final_score'] = compute_final_score(judge_ratings)

df.to_excel(OUTPUT_PATH, index=False)
print(f"\nJudge results saved. Pass rate: {(df['judge_final_score'] == 'pass').sum()}/{len(df)}")

Running judge on all products...
  1/50: Apple iPhone 15 Pro -> fluency=good, grammar=good, tone=good, length=good, grounding=good
  2/50: Samsung Galaxy S24 Ultra -> fluency=good, grammar=good, tone=good, length=good, grounding=good
  3/50: Google Pixel 8 Pro -> fluency=good, grammar=good, tone=good, length=good, grounding=good
  4/50: Sony WH‑1000XM5 Headphones -> fluency=good, grammar=good, tone=good, length=good, grounding=good
  5/50: Bose QuietComfort Ultra Earbuds -> fluency=good, grammar=good, tone=good, length=good, grounding=good
  6/50: Amazon Echo Dot (5th Gen) -> fluency=good, grammar=good, tone=good, length=good, grounding=good
  7/50: Dell XPS 13 9310 Laptop -> fluency=good, grammar=good, tone=good, length=good, grounding=good
  8/50: Apple MacBook Air 13″ (M3) -> fluency=good, grammar=good, tone=good, length=good, grounding=good
  9/50: Microsoft Surface Pro 10 -> fluency=good, grammar=good, tone=good, length=good, grounding=good
  10/50: Garmin Forerunner 255 -> fluenc

In [35]:
# Task 6.3: Compare judge to human evaluation
print("=== Human vs Judge Agreement (on manually rated products) ===")
print()

for criterion in JUDGE_CRITERIA:
    agreements = 0
    total = 0
    for idx in manual_indices:
        human = df.at[idx, criterion]
        judge = df.at[idx, f'judge_{criterion}']
        if human and human != '':  # only count if human rated
            total += 1
            if human == judge:
                agreements += 1
    if total > 0:
        print(f"{criterion:12s}: {agreements}/{total} agree ({agreements/total*100:.0f}%)")

# Show disagreements
print("\n=== Disagreements ===")
for idx in manual_indices:
    for criterion in JUDGE_CRITERIA:
        human = df.at[idx, criterion]
        judge = df.at[idx, f'judge_{criterion}']
        if human and human != '' and human != judge:
            print(f"  Product {idx+1} ({df.at[idx, 'product_name']}): "
                  f"{criterion} human={human} vs judge={judge}")

=== Human vs Judge Agreement (on manually rated products) ===

fluency     : 10/15 agree (67%)
grammar     : 12/15 agree (80%)
tone        : 11/15 agree (73%)
length      : 15/15 agree (100%)
grounding   : 10/15 agree (67%)

=== Disagreements ===
  Product 1 (Apple iPhone 15 Pro): fluency human=ok vs judge=good
  Product 3 (Google Pixel 8 Pro): tone human=ok vs judge=good
  Product 4 (Sony WH‑1000XM5 Headphones): fluency human=ok vs judge=good
  Product 4 (Sony WH‑1000XM5 Headphones): grounding human=ok vs judge=good
  Product 5 (Bose QuietComfort Ultra Earbuds): grammar human=bad vs judge=good
  Product 7 (Dell XPS 13 9310 Laptop): fluency human=bad vs judge=good
  Product 7 (Dell XPS 13 9310 Laptop): tone human=ok vs judge=good
  Product 7 (Dell XPS 13 9310 Laptop): grounding human=ok vs judge=good
  Product 8 (Apple MacBook Air 13″ (M3)): grammar human=ok vs judge=good
  Product 9 (Microsoft Surface Pro 10): fluency human=ok vs judge=good
  Product 9 (Microsoft Surface Pro 10): grou

In [36]:
# Task 6.4: Criterion-by-criterion judging
print("Running criterion-by-criterion judging on manually rated products...")
isolated_results = {c: {} for c in JUDGE_CRITERIA}

for idx in manual_indices:
    product = df_products.iloc[idx].to_dict()
    desc = df.at[idx, 'generated_description']
    print(f"  Product {idx+1}: {product['product_name']}")
    
    for criterion in JUDGE_CRITERIA:
        result = judge_single_criterion(product, desc, criterion)
        isolated_results[criterion][idx] = result['verdict']

print("\n=== Isolated vs Batch Judge Agreement ===")
for criterion in JUDGE_CRITERIA:
    agree_batch = 0
    agree_human = 0
    total = len(manual_indices)
    for idx in manual_indices:
        batch_v = df.at[idx, f'judge_{criterion}']
        isolated_v = isolated_results[criterion][idx]
        human_v = df.at[idx, criterion]
        if batch_v == isolated_v:
            agree_batch += 1
        if human_v and human_v != '' and human_v == isolated_v:
            agree_human += 1
    print(f"{criterion:12s}: batch-agree={agree_batch}/{total} ({agree_batch/total*100:.0f}%)  "
          f"human-agree={agree_human}/{total} ({agree_human/total*100:.0f}%)")

print("\nIsolating criteria may change results because the judge focuses entirely on one")
print("dimension without being influenced by its assessments of other criteria.")

Running criterion-by-criterion judging on manually rated products...
  Product 1: Apple iPhone 15 Pro
  Product 2: Samsung Galaxy S24 Ultra
  Product 3: Google Pixel 8 Pro
  Product 4: Sony WH‑1000XM5 Headphones
  Product 5: Bose QuietComfort Ultra Earbuds
  Product 6: Amazon Echo Dot (5th Gen)
  Product 7: Dell XPS 13 9310 Laptop
  Product 8: Apple MacBook Air 13″ (M3)
  Product 9: Microsoft Surface Pro 10
  Product 10: Garmin Forerunner 255
  Product 11: Fitbit Charge 6
  Product 12: GoPro HERO12 Black
  Product 13: DJI Mini 4 Pro Drone
  Product 14: Nintendo Switch OLED
  Product 15: PlayStation 5 Slim

=== Isolated vs Batch Judge Agreement ===
fluency     : batch-agree=15/15 (100%)  human-agree=10/15 (67%)
grammar     : batch-agree=15/15 (100%)  human-agree=12/15 (80%)
tone        : batch-agree=15/15 (100%)  human-agree=11/15 (73%)
length      : batch-agree=15/15 (100%)  human-agree=15/15 (100%)
grounding   : batch-agree=15/15 (100%)  human-agree=10/15 (67%)

Isolating criteria may

In [37]:
# Task 6.5: Analysis
print("=== Analysis ===")
print()
print("Trade-offs: Human Evaluation vs LLM-as-a-Judge")
print("─" * 60)
print(f"{'Dimension':<20} {'Human':<20} {'LLM Judge':<20}")
print(f"{'Cost':<20} {'High (manual labor)':<20} {'Low (API calls)':<20}")
print(f"{'Scale':<20} {'10-50 items':<20} {'Thousands/day':<20}")
print(f"{'Consistency':<20} {'Variable (fatigue)':<20} {'High (deterministic)':<20}")
print(f"{'Accuracy':<20} {'Gold standard':<20} {'Good but imperfect':<20}")
print(f"{'Nuance':<20} {'Catches subtlety':<20} {'May miss context':<20}")
print()
print("Recommendation for production (thousands daily):")
print("Use LLM-as-a-judge for automated scoring at scale, with periodic human")
print("spot-checks (e.g., 5% sample) to calibrate the judge and catch drift.")
print("Run criterion-by-criterion for higher accuracy on critical dimensions (grounding).")

=== Analysis ===

Trade-offs: Human Evaluation vs LLM-as-a-Judge
────────────────────────────────────────────────────────────
Dimension            Human                LLM Judge           
Cost                 High (manual labor)  Low (API calls)     
Scale                10-50 items          Thousands/day       
Consistency          Variable (fatigue)   High (deterministic)
Accuracy             Gold standard        Good but imperfect  
Nuance               Catches subtlety     May miss context    

Recommendation for production (thousands daily):
Use LLM-as-a-judge for automated scoring at scale, with periodic human
spot-checks (e.g., 5% sample) to calibrate the judge and catch drift.
Run criterion-by-criterion for higher accuracy on critical dimensions (grounding).


In [38]:
# Final save
df.to_excel(OUTPUT_PATH, index=False)
print(f"Final spreadsheet saved to {OUTPUT_PATH}")
print(f"Total columns: {len(df.columns)}")
print(f"Columns: {list(df.columns)}")

Final spreadsheet saved to assignment_01.xlsx
Total columns: 29
Columns: ['product_name', 'Product_attribute_list', 'material', 'warranty', 'generated_description', 'latency_ms', 'input_tokens', 'output_tokens', 'word_count', 'fluency', 'grammar', 'tone', 'length', 'grounding', 'latency', 'cost', 'final_score', 'cost_usd', 'judge_fluency', 'judge_fluency_explanation', 'judge_grammar', 'judge_grammar_explanation', 'judge_tone', 'judge_tone_explanation', 'judge_length', 'judge_length_explanation', 'judge_grounding', 'judge_grounding_explanation', 'judge_final_score']
